In [ ]:
"""
fact_trades — completed
fact_p2p_transactions
fact_balance_snapshots
fact_asset_movements
fact_daily_prices
"""

####Fact trades

In [ ]:
from pyspark.sql.functions import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

binance_trades_df = spark.table(
    f"{catalog}.{silver_schema}.binance_trades"
)

giottus_trades_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_trades"
)

print("Binance trade rows:", binance_trades_df.count())
print("Giottus trade rows:", giottus_trades_df.count())

binance_trades_df.printSchema()
giottus_trades_df.printSchema()

In [ ]:
silver_trades_df = (
    binance_trades_df
    .unionByName(giottus_trades_df)
    .dropDuplicates(["trade_key"])
)

print(
    "Combined Silver trade rows:",
    silver_trades_df.count()
)

display(
    silver_trades_df
    .groupBy("exchange")
    .count()
    .orderBy("exchange")
)

In [ ]:
fact_trades_df = (
    silver_trades_df
    .select(
        col("trade_key"),

        date_format(
            col("trade_time"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        upper(
            col("exchange")
        ).alias("exchange_key"),

        upper(
            col("asset_spent")
        ).alias("spent_asset_key"),

        upper(
            col("asset_received")
        ).alias("received_asset_key"),

        col("transaction_id"),
        col("trade_source"),
        col("symbol"),
        col("side"),

        col("amount_spent"),
        col("amount_received"),

        col("fee_amount"),
        col("fee_asset"),

        col("trade_time"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Gold trade rows:",
    fact_trades_df.count()
)

display(
    fact_trades_df.orderBy(
        col("trade_time").desc()
    )
)

In [ ]:
(
    fact_trades_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_trades"
    )
)

display(
    spark.table(
        f"{catalog}.{gold_schema}.fact_trades"
    )
)

####fact p2p transactions

In [ ]:
silver_p2p_df = spark.table(
    f"{catalog}.{silver_schema}.binance_p2p_transactions"
)

fact_p2p_df = (
    silver_p2p_df
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("P2P"),
            col("p2p_order_id")
        ).alias("transaction_key"),

        date_format(
            col("transaction_time"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(
            col("asset")
        ).alias("asset_key"),

        col("p2p_order_id"),
        col("trade_type"),

        upper(
            col("fiat")
        ).alias('fiat_currency'),

        col("asset_amount"),
        col("fiat_amount"),
        col("unit_price"),
        col("fee_amount"),
        col("net_asset_amount"),

        col("status"),
        col("is_completed"),
        col("transaction_time"),
        col("payment_method"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Gold P2p rows:",
    fact_p2p_df.count()
)

display(
    fact_p2p_df.orderBy(
        col("transaction_time").desc()
    )
)

In [ ]:
(
    fact_p2p_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_p2p_transactions"
    )
)

saved_fact_p2p_df = spark.table(
    f"{catalog}.{gold_schema}.fact_p2p_transactions"
)

print(
    "Saved Gold P2P rows:",
    saved_fact_p2p_df.count()
)

####Fact balance snapshots

In [ ]:
# Load balance snapshots

spot_snapshot_balances_df = spark.table(
    f"{catalog}.{silver_schema}.binance_account_balances"
)

funding_snapshot_balances_df = spark.table(
    f"{catalog}.{silver_schema}.binance_funding_account_balances"
)

giottus_snapshot_balances_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_account_balances"
)


# Prepare Binance Spot balances

spot_fact_source_df = (
    spot_snapshot_balances_df
    .select(
        col("snapshot_id"),
        col("account_id"),
        lit("SPOT").alias("wallet_type"),
        lit("BINANCE_SPOT").alias("wallet_key"),
        upper(col("asset")).alias("asset_key"),
        col("free_balance").cast("decimal(38,18)").alias("free_balance"),
        col("locked_balance").cast("decimal(38,18)").alias("locked_balance"),
        lit(0).cast("decimal(38,18)").alias("frozen_balance"),
        lit(0).cast("decimal(38,18)").alias("withdrawing_balance"),
        col("total_balance").cast("decimal(38,18)").alias("total_balance"),
        upper(col("exchange")).alias("exchange_key"),

        coalesce(
            to_timestamp(
                regexp_extract(
                    col("snapshot_id"),
                    r"account_(\d{8}T\d{6}_\d{6}Z)",
                    1
                ),
                "yyyyMMdd'T'HHmmss_SSSSSS'Z'"
            ),
            col("bronze_ingested_at")
        ).alias("snapshot_time"),

        col("_source_file")
    )
)


# Prepare Binance Funding balances

funding_fact_source_df = (
    funding_snapshot_balances_df
    .select(
        col("snapshot_id"),
        lit(None).cast("string").alias("account_id"),
        upper(col("wallet_type")).alias("wallet_type"),
        lit("BINANCE_FUNDING").alias("wallet_key"),
        upper(col("asset")).alias("asset_key"),
        col("free_balance").cast("decimal(38,18)").alias("free_balance"),
        col("locked_balance").cast("decimal(38,18)").alias("locked_balance"),
        col("frozen_balance").cast("decimal(38,18)").alias("frozen_balance"),
        col("withdrawing_balance")
            .cast("decimal(38,18)")
            .alias("withdrawing_balance"),
        col("total_balance").cast("decimal(38,18)").alias("total_balance"),
        upper(col("exchange")).alias("exchange_key"),

        coalesce(
            col("snapshot_time"),
            col("bronze_ingested_at")
        ).alias("snapshot_time"),

        col("_source_file")
    )
)


# Prepare Giottus Spot balances

giottus_fact_source_df = (
    giottus_snapshot_balances_df
    .select(
        col("snapshot_id"),
        col("account_id"),
        lit("SPOT").alias("wallet_type"),
        lit("GIOTTUS_SPOT").alias("wallet_key"),
        upper(col("asset")).alias("asset_key"),
        col("free_balance").cast("decimal(38,18)").alias("free_balance"),
        col("locked_balance").cast("decimal(38,18)").alias("locked_balance"),
        lit(0).cast("decimal(38,18)").alias("frozen_balance"),
        lit(0).cast("decimal(38,18)").alias("withdrawing_balance"),
        col("total_balance").cast("decimal(38,18)").alias("total_balance"),
        upper(col("exchange")).alias("exchange_key"),
        col("bronze_ingested_at").alias("snapshot_time"),
        col("_source_file")
    )
)


# Combine all wallet snapshots

combined_snapshot_balances_df = (
    spot_fact_source_df
    .unionByName(funding_fact_source_df)
    .unionByName(giottus_fact_source_df)
)


# Keep only assets listed in dim_asset

tracked_assets_df = (
    spark.table(f"{catalog}.{gold_schema}.dim_asset")
    .select("asset_key")
    .dropDuplicates()
)


# Create the combined balance-snapshot fact dataframe

fact_balance_snapshots_df = (
    combined_snapshot_balances_df
    .join(
        tracked_assets_df,
        ["asset_key"],
        "inner"
    )
    .withColumn(
        "date_key",
        date_format(
            col("snapshot_time"),
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "balance_key",
        concat_ws(
            "|",
            col("wallet_key"),
            col("snapshot_id"),
            coalesce(
                col("account_id"),
                lit("NO_ACCOUNT")
            ),
            col("asset_key")
        )
    )
    .dropDuplicates(["balance_key"])
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
    .select(
        "balance_key",
        "date_key",
        "exchange_key",
        "wallet_key",
        "wallet_type",
        "asset_key",
        "snapshot_id",
        "account_id",
        "free_balance",
        "locked_balance",
        "frozen_balance",
        "withdrawing_balance",
        "total_balance",
        "snapshot_time",
        "_source_file",
        "gold_processed_at"
    )
)


print(
    "Combined balance snapshot rows:",
    fact_balance_snapshots_df.count()
)

display(
    fact_balance_snapshots_df
    .groupBy(
        "exchange_key",
        "wallet_key"
    )
    .count()
    .orderBy(
        "exchange_key",
        "wallet_key"
    )
)

display(
    fact_balance_snapshots_df
    .filter(col("exchange_key") == "GIOTTUS")
)

In [ ]:
(
    fact_balance_snapshots_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_balance_snapshots"
    )
)

####Fact asset movements

In [ ]:
gold_trades_df = spark.table(
    f"{catalog}.{gold_schema}.fact_trades"
)

trade_spent_df = (
    gold_trades_df
    .select(
        concat_ws(
            "|",
            col("trade_key"),
            lit("SPENT")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),
        col("spent_asset_key").alias("asset_key"),
        lit("TRADE").alias("movement_type"),
        col("trade_time").alias("movement_time"),

        (-col("amount_spent")).alias(
            "quantity_change"
        )
    )
)

trade_received_df = (
    gold_trades_df
    .select(
        concat_ws(
            "|",
            col("trade_key"),
            lit("RECEIVED")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),
        col("received_asset_key").alias("asset_key"),
        lit("TRADE").alias("movement_type"),
        col("trade_time").alias("movement_time"),

        col("amount_received").alias(
            "quantity_change"
        )
    )
)

trade_movements_df = (
    trade_spent_df
    .unionByName(
        trade_received_df
    )
)


display(
    trade_movements_df.orderBy(
        col("movement_time")
    )
)

In [ ]:
gold_p2p_df = spark.table(
    f"{catalog}.{gold_schema}.fact_p2p_transactions"
)

p2p_movements_df = (
    gold_p2p_df
    .filter(
        col("is_completed") == True
    )
    .select(
        concat_ws(
            "|",
            col("transaction_key"),
            lit("ASSET")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),
        col("asset_key"),

        lit("P2P").alias("movement_type"),

        col("transaction_time").alias(
            "movement_time"
        ),

        when(
            upper(col("trade_type")) == "BUY",
            col("net_asset_amount")
        )
        .otherwise(
            -col("net_asset_amount")
        )
        .alias("quantity_change")
    )
)

print(
    "P2P movement rows:",
    p2p_movements_df.count()
)

display(
    p2p_movements_df.orderBy(
        col("movement_time")
    )
)

In [ ]:
# Binance deposits

binance_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.binance_deposits"
)

binance_deposit_movements_df = (
    binance_deposits_df
    .filter(col("status_code").isin(1, 6))
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias("exchange_key"),
        upper(col("asset")).alias("asset_key"),
        lit("DEPOSIT").alias("movement_type"),
        col("deposit_time").alias("movement_time"),
        col("amount").alias("quantity_change")
    )
)


# Giottus fiat deposits

giottus_fiat_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_fiat_deposits"
)

giottus_fiat_deposit_movements_df = (
    giottus_fiat_deposits_df
    .filter(upper(col("status")) == "SUCCESS")
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("FIAT_DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias("exchange_key"),
        upper(col("asset")).alias("asset_key"),
        lit("DEPOSIT").alias("movement_type"),
        col("deposit_time").alias("movement_time"),

        (
            col("amount")
            - coalesce(
                col("fee_amount"),
                lit(0).cast("decimal(38,18)")
            )
        ).alias("quantity_change")
    )
)


# Giottus crypto deposits

giottus_crypto_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_crypto_deposits"
)

giottus_crypto_deposit_movements_df = (
    giottus_crypto_deposits_df
    .filter(upper(col("status")) == "SUCCESS")
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("CRYPTO_DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias("exchange_key"),
        upper(col("asset")).alias("asset_key"),
        lit("DEPOSIT").alias("movement_type"),
        col("deposit_time").alias("movement_time"),

        (
            col("amount")
            - coalesce(
                col("fee_amount"),
                lit(0).cast("decimal(38,18)")
            )
        ).alias("quantity_change")
    )
)


# Combine all deposits

deposit_movements_df = (
    binance_deposit_movements_df
    .unionByName(giottus_fiat_deposit_movements_df)
    .unionByName(giottus_crypto_deposit_movements_df)
)


print(
    "Deposit movement rows:",
    deposit_movements_df.count()
)

display(
    deposit_movements_df
    .groupBy("exchange_key")
    .count()
    .orderBy("exchange_key")
)

display(
    deposit_movements_df.orderBy("movement_time")
)

In [ ]:
# Binance withdrawals

binance_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.binance_withdrawals"
)

binance_withdrawal_movements_df = (
    binance_withdrawals_df
    .filter(col("status_code") == 6)
    .withColumn(
        "movement_timestamp",
        coalesce(
            col("completed_at"),
            col("requested_at")
        )
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("movement_timestamp"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias("exchange_key"),
        upper(col("asset")).alias("asset_key"),
        lit("WITHDRAWAL").alias("movement_type"),
        col("movement_timestamp").alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("transaction_fee"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# Giottus fiat withdrawals

giottus_fiat_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_fiat_withdrawals"
)

giottus_fiat_withdrawal_movements_df = (
    giottus_fiat_withdrawals_df
    .filter(upper(col("status")) == "SUCCESS")
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("FIAT_WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("withdrawal_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias("exchange_key"),
        upper(col("asset")).alias("asset_key"),
        lit("WITHDRAWAL").alias("movement_type"),
        col("withdrawal_time").alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("fee_amount"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# Giottus crypto withdrawals

giottus_crypto_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_crypto_withdrawals"
)

giottus_crypto_withdrawal_movements_df = (
    giottus_crypto_withdrawals_df
    .filter(upper(col("status")) == "SUCCESS")
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("CRYPTO_WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("withdrawal_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias("exchange_key"),
        upper(col("asset")).alias("asset_key"),
        lit("WITHDRAWAL").alias("movement_type"),
        col("withdrawal_time").alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("transaction_fee"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# Combine all withdrawals

withdrawal_movements_df = (
    binance_withdrawal_movements_df
    .unionByName(giottus_fiat_withdrawal_movements_df)
    .unionByName(giottus_crypto_withdrawal_movements_df)
)


print(
    "Withdrawal movement rows:",
    withdrawal_movements_df.count()
)

display(
    withdrawal_movements_df
    .groupBy("exchange_key")
    .count()
    .orderBy("exchange_key")
)

display(
    withdrawal_movements_df.orderBy("movement_time")
)

In [ ]:
silver_transfers_df = spark.table(
    f"{catalog}.{silver_schema}.binance_transfers"
)

transfer_movements_df = (
    silver_transfers_df
    .filter(
        upper(col("status")).isin(
            "CONFIRMED",
            "SUCCESS"
        )
    )
    .withColumn(
        "transfer_quantity_change",

        when(
            upper(
                col("requested_transfer_type")
            ).startswith("MAIN_"),

            -col("amount")
        )
        .when(
            upper(
                col("requested_transfer_type")
            ).endswith("_MAIN"),

            col("amount")
        )
        .otherwise(
            lit(0).cast("decimal(38,18)")
        )
    )
    .filter(
        col("transfer_quantity_change") != 0
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("TRANSFER"),
            col("transfer_id")
        ).alias("movement_key"),

        date_format(
            col("transfer_time"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        upper(col("exchange")).alias(
            "exchange_key"
        ),

        upper(col("asset")).alias(
            "asset_key"
        ),

        lit("TRANSFER").alias(
            "movement_type"
        ),

        col("transfer_time").alias(
            "movement_time"
        ),

        col("transfer_quantity_change").alias(
            "quantity_change"
        )
    )
)

print(
    "Transfer movement rows:",
    transfer_movements_df.count()
)

display(transfer_movements_df)

In [ ]:
trade_fee_movements_df = (
    gold_trades_df
    .filter(
        col("fee_amount").isNotNull()
        & (col("fee_amount") > 0)
    )
    .select(
        concat_ws(
            "|",
            col("trade_key"),
            lit("FEE")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),

        upper(
            col("fee_asset")
        ).alias("asset_key"),

        lit("TRADE_FEE").alias(
            "movement_type"
        ),

        col("trade_time").alias(
            "movement_time"
        ),

        (
            -col("fee_amount")
        ).alias(
            "quantity_change"
        )
    )
)


print(
    "Trade fee movement rows:",
    trade_fee_movements_df.count()
)


display(
    trade_fee_movements_df.orderBy(
        col("movement_time")
    )
)

In [ ]:
all_asset_movements_df = (
    trade_movements_df

    .unionByName(
        p2p_movements_df
    )

    .unionByName(
        deposit_movements_df
    )

    .unionByName(
        withdrawal_movements_df
    )

    .unionByName(
        transfer_movements_df
    )

    .unionByName(
        trade_fee_movements_df
    )
)

fact_asset_movements_df = (
    all_asset_movements_df
    .filter(
        col("movement_key").isNotNull()
        & col("asset_key").isNotNull()
        & col("movement_time").isNotNull()
        & col("quantity_change").isNotNull()
    )
    .withColumn(
        "quantity_change",
        col("quantity_change")
        .cast("decimal(38,18)")
    )
    .dropDuplicates([
        "movement_key"
    ])
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Total asset movement rows:",
    fact_asset_movements_df.count()
)

display(
    fact_asset_movements_df
    .groupBy("movement_type")
    .count()
    .orderBy("movement_type")
)

display(
    fact_asset_movements_df.orderBy(
        col("movement_time")
    )
)

In [ ]:
(
    fact_asset_movements_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_asset_movements"
    )
)

saved_movements_df = spark.table(
    f"{catalog}.{gold_schema}.fact_asset_movements"
)

print(
    "Saved asset movement rows:",
    saved_movements_df.count()
)

display(
    saved_movements_df
    .groupBy("movement_type")
    .count()
    .orderBy("movement_type")
)

####Fact daily balances

In [ ]:
balance_snapshots_df = spark.table(
    f"{catalog}.{gold_schema}.fact_balance_snapshots"
)


# Find the latest snapshot time for each wallet

latest_snapshot_times_df = (
    balance_snapshots_df
    .groupBy(
        "exchange_key",
        "wallet_key"
    )
    .agg(
        max("snapshot_time").alias(
            "latest_snapshot_time"
        )
    )
)


# Get every asset from each latest wallet snapshot

latest_wallet_balances_df = (
    balance_snapshots_df.alias("balances")
    .join(
        latest_snapshot_times_df.alias("latest"),
        (
            (
                col("balances.exchange_key")
                == col("latest.exchange_key")
            )
            & (
                col("balances.wallet_key")
                == col("latest.wallet_key")
            )
            & (
                col("balances.snapshot_time")
                == col("latest.latest_snapshot_time")
            )
        ),
        "inner"
    )
    .select(
        col("balances.exchange_key"),
        col("balances.asset_key").alias("asset"),
        col("balances.total_balance")
    )
    .filter(
        col("total_balance") > 0
    )
)


# Combine wallets belonging to the same exchange

latest_total_balances_df = (
    latest_wallet_balances_df
    .groupBy(
        "exchange_key",
        "asset"
    )
    .agg(
        sum("total_balance").alias(
            "current_balance"
        )
    )
)


print(
    "Latest total balance rows:",
    latest_total_balances_df.count()
)

display(
    latest_total_balances_df.orderBy(
        "exchange_key",
        "asset"
    )
)

In [ ]:
movements_df = spark.table(
    f"{catalog}.{gold_schema}.fact_asset_movements"
)


daily_movements_df = (
    movements_df
    .withColumn(
        "movement_date",
        to_date(col("movement_time"))
    )
    .groupBy(
        "movement_date",
        "exchange_key",
        "asset_key"
    )
    .agg(
        sum("quantity_change").alias(
            "daily_change"
        )
    )
)


print(
    "Daily movement rows:",
    daily_movements_df.count()
)


display(
    daily_movements_df.orderBy(
        "movement_date",
        "exchange_key",
        "asset_key"
    )
)

In [ ]:
date_df = spark.table(
    f"{catalog}.{gold_schema}.dim_date"
)
date_df.printSchema()
print("total date rows: ", date_df.count())

In [ ]:
date_range_df = (
    date_df
    .agg(
        min("full_date").alias("start_date"),
        max("full_date").alias("end_date")
    )
)

display(date_range_df)

In [ ]:
assets_df = spark.table(
    f"{catalog}.{gold_schema}.dim_asset"
)
assets_df.printSchema()
print("Asset rows: ",assets_df.count())

In [ ]:
balance_dates_df = (
    date_df
    .select(
        col("full_date").alias(
            "balance_date"
        )
    )
)


current_exchange_assets_df = (
    latest_total_balances_df
    .select(
        col("exchange_key"),
        upper(col("asset")).alias(
            "asset_key"
        )
    )
)


movement_exchange_assets_df = (
    daily_movements_df
    .select(
        "exchange_key",
        "asset_key"
    )
)


balance_exchange_assets_df = (
    current_exchange_assets_df
    .unionByName(
        movement_exchange_assets_df
    )
    .dropDuplicates([
        "exchange_key",
        "asset_key"
    ])
)


balance_calendar_df = (
    balance_dates_df
    .crossJoin(
        balance_exchange_assets_df
    )
)


print(
    "Balance calendar rows:",
    balance_calendar_df.count()
)


display(
    balance_calendar_df.orderBy(
        "exchange_key",
        "asset_key",
        "balance_date"
    )
)

In [ ]:
daily_changes_df = (
    daily_movements_df
    .withColumnRenamed(
        "movement_date",
        "balance_date"
    )
)


balance_activity_df = (
    balance_calendar_df
    .join(
        daily_changes_df,
        [
            "balance_date",
            "exchange_key",
            "asset_key"
        ],
        "left"
    )
    .fillna(
        0,
        subset=["daily_change"]
    )
)


print(
    "Balance activity rows:",
    balance_activity_df.count()
)

In [ ]:
current_balances_df = (
    latest_total_balances_df
    .select(
        col("exchange_key"),

        upper(col("asset")).alias(
            "asset_key"
        ),

        col("current_balance")
    )
)


balance_base_df = (
    balance_activity_df
    .join(
        current_balances_df,
        [
            "exchange_key",
            "asset_key"
        ],
        "left"
    )
    .fillna(
        0,
        subset=["current_balance"]
    )
)


print(
    "Balance base rows:",
    balance_base_df.count()
)

In [ ]:
from pyspark.sql.window import Window


future_movements_window = (
    Window
    .partitionBy(
        "exchange_key",
        "asset_key"
    )
    .orderBy(
        "balance_date"
    )
    .rowsBetween(
        1,
        Window.unboundedFollowing
    )
)


balance_with_future_df = (
    balance_base_df
    .withColumn(
        "movements_after_date",
        coalesce(
            sum("daily_change").over(
                future_movements_window
            ),
            lit(0).cast(
                "decimal(38,18)"
            )
        )
    )
)


display(
    balance_with_future_df
    .select(
        "balance_date",
        "exchange_key",
        "asset_key",
        "daily_change",
        "current_balance",
        "movements_after_date"
    )
    .orderBy(
        "exchange_key",
        "asset_key",
        "balance_date"
    )
)

In [ ]:
calculated_balances_df = (
    balance_with_future_df
    .withColumn(
        "balance_amount",
        (
            col("current_balance")
            - col("movements_after_date")
        )
    )
)

print(
    "Calculated rows:",
    calculated_balances_df.count()
)

print(
    "Negative balance rows:",
    calculated_balances_df
    .filter(col("balance_amount") < 0)
    .count()
)
display(
    calculated_balances_df
    .orderBy(
        col("balance_date").asc()
    )
)

In [ ]:
fact_daily_balances_df = (
    calculated_balances_df
    .withColumn(
        "date_key",
        date_format(
            col("balance_date"),
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "balance_key",
        concat_ws(
            "|",
            col("exchange_key"),
            col("asset_key"),
            col("date_key")
        )
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
    .select(
        "balance_key",
        "date_key",
        "balance_date",
        "exchange_key",
        "asset_key",
        "balance_amount",
        "daily_change",
        "gold_processed_at"
    )
)


fact_daily_balances_df.printSchema()

In [ ]:
duplicate_keys = (
    fact_daily_balances_df
    .groupBy("balance_key")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate balance keys:", duplicate_keys)

In [ ]:
(
    fact_daily_balances_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_daily_balances"
    )
)

saved_daily_balances_df = spark.table(
    f"{catalog}.{gold_schema}.fact_daily_balances"
)

print(
    "Saved daily balance rows:",
    saved_daily_balances_df.count()
)

####Fact daily prices

In [ ]:
silver_price_history = (
    spark.table(
        f"{catalog}.{silver_schema}.binance_price_history"
    )
)

print("silver price history rows: ",silver_price_history.count())

silver_price_history.printSchema()

display(
    silver_price_history
    .orderBy(
        col("open_time").desc()
    )
    .limit(5)
)


In [ ]:
price_keys_df = (
    silver_price_history
    .filter(
        col("is_closed") == True
    )
    .withColumn(
        "date_key",
        date_format(
            col("open_time"),
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "exchange_key",
        upper(
            col("exchange")
        )
    )
    .withColumn(
        "asset_key",
        upper(
            col("asset")
        )
    )
    .withColumn(
        "quote_asset_key",
        lit("USDT")
    )
)


print(
    "Closed price rows:",
    price_keys_df.count()
)


display(
    price_keys_df
    .select(
        "date_key",
        "exchange_key",
        "asset_key",
        "quote_asset_key",
        "symbol",
        "open_time"
    )
    .orderBy(
        col("open_time").desc()
    )
    .limit(5)
)

In [ ]:
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    upper
)

fact_daily_prices_df = (
    price_keys_df
    .withColumn(
        "price_key",
        concat_ws(
            "|",
            col("exchange_key"),
            upper(col("symbol")),
            col("date_key").cast("string")
        )
    )
    .select(
        "price_key",
        "date_key",
        "exchange_key",
        "asset_key",
        "quote_asset_key",
        "symbol",
        "open_time",
        "close_time",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "volume",
        "quote_asset_volume",
        "trade_count",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "price_change",
        "price_change_pct",
        "_source_file",
        "bronze_ingested_at"
    )
    .dropDuplicates([
        "price_key"
    ])
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Gold daily price rows:",
    fact_daily_prices_df.count()
)

display(
    fact_daily_prices_df
    .orderBy(
        col("open_time").desc()
    )
    .limit(5)
)

In [ ]:
total_rows = fact_daily_prices_df.count()
distinct = (
    fact_daily_prices_df
    .select(
        "price_key"
    )
    .distinct()
    .count()
)

duplicate = total_rows - distinct

print(
    "Total rows: ", total_rows,
    "Distinct rows: ", distinct,
    "Duplicates: ", duplicate
)

In [ ]:
(
    fact_daily_prices_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema",True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_daily_prices"
    )
)

####Fact daily portfolio

In [ ]:
daily_balances_df = (
    spark.table(
        f"{catalog}.{gold_schema}.fact_daily_balances"
    )
)

daily_prices_df = (
    spark.table(
        f"{catalog}.{gold_schema}.fact_daily_prices"
    )
)

print("Daily balances rows: ", daily_balances_df.count())
print("Daily prices rows: ", daily_prices_df.count())

daily_balances_df.printSchema()
daily_prices_df.printSchema()

In [ ]:
from pyspark.sql.window import Window


# Use Binance USDT prices as the reference market prices

reference_prices_df = (
    daily_prices_df
    .filter(
        (col("exchange_key") == "BINANCE")
        & (col("quote_asset_key") == "USDT")
    )
    .select(
        "date_key",
        "asset_key",
        "close_price"
    )
    .dropDuplicates([
        "date_key",
        "asset_key"
    ])
)


# Join prices by date and asset, not by the wallet's exchange

portfolio_base_df = (
    daily_balances_df
    .join(
        reference_prices_df,
        on=[
            "date_key",
            "asset_key"
        ],
        how="left"
    )
)


# Carry the latest available price forward

latest_price_window = (
    Window
    .partitionBy(
        "exchange_key",
        "asset_key"
    )
    .orderBy("date_key")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)


portfolio_base_df = (
    portfolio_base_df
    .withColumn(
        "close_price",
        last(
            "close_price",
            ignorenulls=True
        ).over(latest_price_window)
    )
)


print(
    "Portfolio rows:",
    portfolio_base_df.count()
)

print(
    "Rows without reference prices:",
    portfolio_base_df
    .filter(col("close_price").isNull())
    .count()
)

display(
    portfolio_base_df
    .filter(col("close_price").isNull())
    .select(
        "exchange_key",
        "asset_key"
    )
    .distinct()
    .orderBy(
        "exchange_key",
        "asset_key"
    )
)

In [ ]:
total_rows = portfolio_base_df.count()

missing_price_rows = (
    portfolio_base_df
    .filter(
        col("close_price").isNull()
    )
    .count()
)

print("Portfolio rows:", total_rows)
print("Rows with missing prices:", missing_price_rows)

display(
    portfolio_base_df
    .filter(
        col("close_price").isNull()
    )
    .select("asset_key")
    .distinct()
)

In [ ]:
portfolio_valued_df = (
    portfolio_base_df
    .withColumn(
        "valuation_price",
        when(
            col("asset_key") == "USDT",
            lit(1).cast("decimal(38,18)")
        ).otherwise(
            col("close_price")
        )
    )
    .withColumn(
        "asset_value_usdt",
        (
            col("balance_amount") *
            col("valuation_price")
        ).cast("decimal(38,18)")
    )
)

total_rows = portfolio_valued_df.count()

missing_prices_rows = (
    portfolio_valued_df
    .filter(
        col("valuation_price").isNull()
    )
    .count()
)

negative_prices_rows = (
    portfolio_valued_df
    .filter(
        col("valuation_price") <0
    )
    .count()
)

print("Portfolio rows:", total_rows)
print("Rows with missing prices:", missing_prices_rows)
print("Rows with negative prices:", negative_prices_rows)

In [ ]:
fact_daily_portfolio = (
    portfolio_valued_df
    .withColumn(
        "portfolio_key",
        concat_ws(
            "|",
            "exchange_key",
            "asset_key",
            "date_key"
        )
    )
    .select(
        "portfolio_key",
        "date_key",
        "balance_date",
        "exchange_key",
        "asset_key",
        "balance_amount",
        "daily_change",
        "valuation_price",
        "asset_value_usdt"
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
    .dropDuplicates(
        ["portfolio_key"]
    )
)

total_rows = fact_daily_portfolio.count()

distinct_rows = (
    fact_daily_portfolio
    .select("portfolio_key")
    .distinct()
    .count()
)

print("Daily portfolio rows:", total_rows)
print("Distinct portfolio keys:", distinct_rows)
print("Duplicates rows:", total_rows - distinct_rows)

display(
    fact_daily_portfolio
    .orderBy(
        col("balance_date").desc(),
        col("asset_value_usdt").desc()
    )
    .limit(10)
)

In [ ]:
(
    fact_daily_portfolio
    .write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{gold_schema}.fact_daily_portfolio")
)

In [ ]:
display(spark.sql("""
SELECT asset, total_balance
FROM crypto_lakehouse.silver.binance_account_balances
WHERE snapshot_id = (
    SELECT MAX(snapshot_id)
    FROM crypto_lakehouse.silver.binance_account_balances
)
AND total_balance > 0
ORDER BY asset
"""))

In [ ]:
display(spark.sql("""
SELECT asset_key AS asset, balance_amount
FROM crypto_lakehouse.gold.fact_daily_balances
WHERE balance_date = (
    SELECT MAX(balance_date)
    FROM crypto_lakehouse.gold.fact_daily_balances
)
ORDER BY asset
"""))